In [1]:
# importing libraries and packages

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [2]:
# Load cleaned data from wrangling step

df= pd.read_csv('data/clean/hospital_readmission_clean_v1.csv')

df.head()

,medical_specialty,payer_code,race,diag_3,diag_2,diag_1,patient_nbr,time_in_hospital,admission_source_id,num_lab_procedures,...,glimepiride-pioglitazone_dose_change,metformin-rosiglitazone_flag,metformin-rosiglitazone_dose,metformin-rosiglitazone_dose_change,metformin-pioglitazone_flag,metformin-pioglitazone_dose,metformin-pioglitazone_dose_change,insulin_flag,insulin_dose,insulin_dose_change
0,Pediatrics-Endocrinology,Unknown,Caucasian,Unknown,Unknown,250.83,8222157,1,1,41,...,0,0,0,0,0,0,0,0,0,0
1,Unknown,Unknown,Caucasian,255,250.01,276,55629189,3,7,59,...,0,0,0,0,0,0,0,1,2,1
2,Unknown,Unknown,AfricanAmerican,V27,250,648,86047875,2,7,11,...,0,0,0,0,0,0,0,0,0,0
3,Unknown,Unknown,Caucasian,403,250.43,8,82442376,2,7,44,...,0,0,0,0,0,0,0,1,2,1
4,Unknown,Unknown,Caucasian,250,157,197,42519267,1,7,51,...,0,0,0,0,0,0,0,1,1,0


In [3]:
# View the shape, info 

print(f'The number of rows is {df.shape[0]} and the number of columns is {df.shape[1]}')
print(df.info())

# Show all column names(numbered) and disable truncation for detailed feature review and subsequent classification
pd.set_option('display.max_seq_items', None)

for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

The number of rows is 99343 and the number of columns is 114
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 99343 entries, 0 to 99342
Columns: 114 entries, medical_specialty to insulin_dose_change
dtypes: int64(78), object(36)
memory usage: 86.4+ MB
None
1. medical_specialty
2. payer_code
3. race
4. diag_3
5. diag_2
6. diag_1
7. patient_nbr
8. time_in_hospital
9. admission_source_id
10. num_lab_procedures
11. encounter_id
12. admission_type_id
13. discharge_disposition_id
14. gender
15. age
16. number_inpatient
17. number_emergency
18. number_outpatient
19. num_medications
20. num_procedures
21. number_diagnoses
22. metformin
23. repaglinide
24. nateglinide
25. chlorpropamide
26. glimepiride
27. acetohexamide
28. glipizide
29. glyburide
30. tolbutamide
31. pioglitazone
32. rosiglitazone
33. acarbose
34. miglitol
35. troglitazone
36. tolazamide
37. insulin
38. glyburide-metformin
39. glipizide-metformin
40. glimepiride-pioglitazone
41. metformin-rosiglitazone
42. metformin-pioglitazo

In [4]:
# Reviewing the columns reveals that old Diagonis and engineered Diagnosis Encoding (3-Digit ICD Extraction) exist

df[['diag_1','diag_1_3digit', 'diag_2','diag_2_3digit','diag_3','diag_3_3digit']].head()

,diag_1,diag_1_3digit,diag_2,diag_2_3digit,diag_3,diag_3_3digit
0,250.83,250,Unknown,Unknown,Unknown,Unknown
1,276,276,250.01,250,255,255
2,648,648,250,250,V27,V27
3,8,8,250.43,250,403,403
4,197,197,157,157,250,250


In [5]:
# Drop the raw old Diagnosis data and keep the engineered Diagonis Encoding (3-Digit ICD Extraction) 

df = df.drop(['diag_1', 'diag_2', 'diag_3'], axis=1)

In [6]:
# Our target variable from dataset

target = 'readmit_30d'
df[target].value_counts()

readmit_30d
0    88029
1    11314
Name: count, dtype: int64

In [7]:
# Feature classification (to guide correct encoding, training and testing)

# Drop irrelevant/redundant features
drop_features = [
    'encounter_id',
    'patient_nbr',
    'readmitted',

    'payer_code',  # high cardinality with limited predictive relevance

    # Drop redundant diagnosis columns
    'diag_2_3digit',  # redundant high-cardinality feature; primary diagnosis retained
    'diag_3_3digit',  # redundant high-cardinality feature; adds noise

    # Raw drug variables replaced with engineered features
    'metformin', 'repaglinide', 'nateglinide', 'chlorpropamide',
    'glimepiride', 'acetohexamide', 'glipizide', 'glyburide',
    'tolbutamide', 'pioglitazone', 'rosiglitazone',
    'acarbose', 'miglitol', 'troglitazone',
    'tolazamide', 'insulin', 'glyburide-metformin',
    'glipizide-metformin', 'glimepiride-pioglitazone',
    'metformin-rosiglitazone', 'metformin-pioglitazone'
]

Redundant identifiers, weak predictors, and overlapping diagnosis features were removed. Engineered drug variables were retained instead of raw drug categories.

In [8]:
# Removing irrelevant/redundant features

df = df.drop(columns=drop_features) # 30 columns were dropped after dropping drop_features from our dataset

print(f'The number of rows is {df.shape[0]} and the number of columns is {df.shape[1]}') # Number of columns changed from 114 to 84

The number of rows is 99343 and the number of columns is 84


In [9]:
# Features and target splittiing 

X = df.drop(target, axis=1)
y = df[target]

In [10]:
# Train-Test-Split 
# We are splitting before preprocessing to prevent data leakage and ensure that transformations are learned only from training data

X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2, random_state=42, stratify=y)

In [11]:
# Feature Classification (EDA-Driven)

# Categorical features
categorical_features = [
    'medical_specialty',  # high cardinality but clinically relevant → will be grouped
    'race',
    'gender',
    'age_range',
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id',
    'change',
    'diabetesMed',
    'diag_1_3digit'
]

# Numerical features
numeric_features = [
    'time_in_hospital',
    'num_lab_procedures',
    'num_procedures',
    'num_medications',
    'number_outpatient',
    'number_emergency',
    'number_inpatient',
    'age_mid'
]

# Ordinal feature
ordinal_features = ['number_diagnoses']

# Engineered drug features
drug_flag_features = [col for col in X_train.columns if '_flag' in col]
drug_dose_features = [col for col in X_train.columns if '_dose' in col and 'change' not in col]
drug_change_features = [col for col in X_train.columns if '_dose_change' in col]

Feature classification is based on domain understanding from EDA.

In [12]:
# Handling High-Cardinality features on training data only

# Diagnosis Grouping
top_diag = X_train['diag_1_3digit'].value_counts().nlargest(20).index

X_train['diag_1_3digit'] = X_train['diag_1_3digit'].apply(
    lambda x: x if x in top_diag else 'Other'
)

X_test['diag_1_3digit'] = X_test['diag_1_3digit'].apply(
    lambda x: x if x in top_diag else 'Other'
)

# Medical Specialty Grouping
top_specialties = X_train['medical_specialty'].value_counts().nlargest(10).index

X_train['medical_specialty'] = X_train['medical_specialty'].apply(
    lambda x: x if x in top_specialties else 'Other'
)

X_test['medical_specialty'] = X_test['medical_specialty'].apply(
    lambda x: x if x in top_specialties else 'Other'
)

High-cardinality features were reduced using training data to prevent feature explosion and improve generalization.

In [13]:
# Final Grouping of categorical and numerical columns

categorical_features = categorical_features + drug_change_features
numeric_features = numeric_features + drug_dose_features + drug_flag_features

Feature selection and classification were guided by prior EDA and feature engineering steps. Raw drug variables were replaced with engineered features (flags, dose, and dose change) to improve signal quality and reduce redundancy. All remaining features were explicitly categorized into categorical, numerical, and ordinal groups to ensure appropriate preprocessing.

In [14]:
# Encoding (Categorical Variables)

X_train = pd.get_dummies(X_train, columns=categorical_features, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_features, drop_first=True)

X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

One-hot encoding is applied to categorical variables. The datasets are aligned to ensure consistent feature structure.

In [15]:
# Scaling numeric and ordinal features
scale_cols = numeric_features + ordinal_features

scaler = StandardScaler()

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols] = scaler.transform(X_test[scale_cols])

Standardization ensures all features have comparable scale. The scaler is fit only on training data to prevent data leakage.

In [16]:
# # Quick sanity check: dataset shapes and sample rows
print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

X_train.sample(10)

Train shape: (79474, 157)
Test shape: (19869, 157)


,time_in_hospital,num_lab_procedures,age,number_inpatient,number_emergency,number_outpatient,num_medications,num_procedures,number_diagnoses,age_mid,...,glimepiride_dose_change_1,glipizide_dose_change_1,glyburide_dose_change_1,pioglitazone_dose_change_1,rosiglitazone_dose_change_1,acarbose_dose_change_1,miglitol_dose_change_1,tolazamide_dose_change_1,glyburide-metformin_dose_change_1,insulin_dose_change_1
52008,-1.138209,1.635680,[50-60),-0.498835,-0.207036,-0.289622,-0.493141,-0.784188,-0.726298,-0.674652,...,False,False,False,False,False,False,False,False,False,True
7482,-0.129132,-1.063437,[70-80),-0.498835,-0.207036,-0.289622,0.123148,-0.198262,-1.758233,0.580038,...,False,False,False,False,False,False,False,False,False,False
7016,0.207227,-1.165291,[50-60),-0.498835,-0.207036,-0.289622,-0.123368,-0.198262,-1.242266,-0.674652,...,False,False,False,False,False,False,False,False,False,False
48823,1.216304,0.820852,[60-70),-0.498835,-0.207036,-0.289622,0.369663,-0.198262,0.821605,-0.047307,...,False,False,False,False,False,False,False,False,False,False
97531,0.543586,1.330119,[50-60),-0.498835,-0.207036,-0.289622,0.123148,-0.198262,0.821605,-0.674652,...,False,False,False,False,False,False,False,False,False,False
60999,2.225381,0.871779,[80-90),-0.498835,0.821339,-0.289622,0.862693,-0.784188,0.821605,1.207384,...,False,False,False,False,False,False,False,False,False,True
4000,-0.465491,0.668072,[70-80),-0.498835,-0.207036,-0.289622,-0.493141,-0.784188,0.821605,0.580038,...,False,False,False,False,False,False,False,False,False,False
48180,-0.129132,1.431973,[60-70),-0.498835,-0.207036,-0.289622,0.739436,-0.784188,0.821605,-0.047307,...,False,False,False,False,False,False,False,False,False,True
50544,-0.465491,0.006024,[60-70),-0.498835,-0.207036,-0.289622,1.478981,-0.784188,0.821605,-0.047307,...,False,False,False,False,False,False,False,False,False,False
78583,0.879945,1.839387,[70-80),-0.498835,-0.207036,0.488441,2.095270,-0.198262,0.305637,0.580038,...,False,False,False,False,False,False,False,False,False,True


In [17]:
# Save processed datasets for modeling phase

X_train.to_csv("X_train_processed.csv", index=False)
X_test.to_csv("X_test_processed.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

The processed datasets are saved to ensure reproducibility and to separate preprocessing from the modeling phase.

##  Summary

- Irrelevant and redundant features were removed early, including identifiers and weak predictors  
- Secondary diagnosis variables were dropped due to redundancy, retaining only the primary diagnosis  
- High-cardinality features (diagnosis and medical specialty) were grouped using training data  
- Engineered drug features were retained to capture treatment patterns  
- Categorical variables were encoded using one-hot encoding  
- Numerical and ordinal features were standardized using StandardScaler  
- Data was split before preprocessing to prevent data leakage  
- Final datasets are optimized, reduced in dimensionality, and ready for machine learning modeling  